# 혜진 v4: 변수 선별·중복 제거·암종 변이 서명
Run All로 EDA → 학습 폴드 안에서 변수 선별/상관 제거 → 하이퍼파라미터 비교 → 기존 홀드아웃 재평가 → 전체 재학습 → 피클·새 제출·상세 보고서를 생성합니다. 다른 프로젝트 .py 파일이 필요하지 않습니다.

**입력 변수 수**와 **하이퍼파라미터 수**는 다릅니다. v3는 모델당 입력 21,448개를 사용했습니다. v4에서는 직접 분류기 입력을 64·128·256·512·1,024·2,048개로 줄이거나, 암종별 서명 점수 26개(+선택 변수 128개)로 압축합니다. 선택 개수 k, 연관성 점수, 중복 제거 임계값, 규제 강도·트리 복잡도를 비교합니다.

기존 실제 Public: v1 **0.2765425197**, v3 **0.2998780222**. 새 Public은 아직 알 수 없습니다. 기존 홀드아웃은 여러 차례 관측됐으므로 새로운 독립 평가가 아닙니다. 후보 선택은 개발 그룹 CV의 clean/합성 스트레스만으로 합니다. 테스트로 사전·변수 선별·상관·서명·하이퍼파라미터를 적합하지 않습니다.

필요 패키지: numpy, pandas, scipy, scikit-learn, lightgbm, matplotlib, cloudpickle, threadpoolctl, IPython. macOS LightGBM에는 libomp가 필요할 수 있습니다. 공용 data/에 CSV 세 개를 두고 저장소 루트 또는 models/혜진/에서 실행합니다. 결과는 artifacts/v4/, 제출은 공용 submission/혜진_submission_v4.csv에 저장합니다.

초기 80개 설정의 개발 CV에서 64/128개 축소가 정보를 잃는 경향을 확인해, 1,024/2,048개·요약 유지 대조군·L1 내장 선별 61개를 추가했습니다. 이는 같은 개발 데이터의 순차 탐색이므로 선택 편향이 있습니다. 아래 코드는 Run All 시 최종 전체 141개 설정을 처음부터 평가합니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2,f_classif
from sklearn.preprocessing import StandardScaler,MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display,FileLink
threadpool_limits(2)
SEED=42
roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in roots if all((p/'data'/f).is_file() for f in ['train.csv','test.csv','sample_submission.csv'])),None)
if REPO is None:raise FileNotFoundError('공用 data/에 train.csv, test.csv, sample_submission.csv가 필요합니다.')
DATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v4';OUT.mkdir(parents=True,exist_ok=True)
SUBMIT=REPO/'submission';SUBMIT.mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
train=pd.read_csv(DATA/'train.csv');test=pd.read_csv(DATA/'test.csv');sample=pd.read_csv(DATA/'sample_submission.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert list(test)==['ID']+genes and sample.ID.equals(test.ID)
assert train.ID.is_unique and test.ID.is_unique
y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)
X=train[genes];groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
folds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('DATA',DATA,'OUTPUT',OUT,'Train/test',train.shape,test.shape,'Dev/holdout',len(dev),len(hold))


DATA /Users/mac/project/cancer_project/data OUTPUT /Users/mac/project/cancer_project/artifacts/v4 Train/test (6201, 4386) (2546, 4385) Dev/holdout 4960 1241


## 1. EDA 및 고정 파서
v3의 X→* 종결 정규화와 동일한 손실 없는 행별 파생변수 생성기를 사용합니다. 아래 파서는 집단 통계나 정답을 사용하지 않습니다. gene/variant/position/type, 환자 요약 18종, 수작업 그룹 7종을 먼저 후보로 만들며, 이후 폴드 내에서 선택합니다. 비교 기준만 전체 변수를 쓰고, compact 후보는 주석 수에 민감한 numeric 전역 집계를 선택 풀에서 제외합니다. 변이 유전자 수는 임상 TMB가 아닙니다.

In [2]:
GROUPS = {
 'p53_cellcycle': 'TP53 RB1 CDKN2A CDKN2B CDK4 CDK6 CCND1 ATM CHEK2'.split(),
 'rtk_ras_mapk': 'EGFR ERBB2 ERBB3 KRAS NRAS HRAS BRAF NF1 MAP2K1 MAP2K2 KIT PDGFRA FGFR2 FGFR3'.split(),
 'pi3k_mtor': 'PIK3CA PIK3R1 PTEN AKT1 MTOR TSC1 TSC2 STK11'.split(),
 'dna_repair': 'BRCA1 BRCA2 PALB2 ATM ATR CHEK2 MLH1 MSH2 MSH6 PMS2 POLE POLD1'.split(),
 'chromatin': 'ARID1A ARID1B ARID2 PBRM1 SMARCA4 BAP1 SETD2 KMT2A KMT2C KMT2D ATRX'.split(),
 'wnt': 'APC CTNNB1 AXIN1 AXIN2 RNF43'.split(),
 'metabolism': 'IDH1 IDH2 SDHB SDHC SDHD FH VHL'.split(),
}

def kind(token):
 if 'fs' in token.lower(): return 'frameshift'
 if '*' in token: return 'stop'
 if 'del' in token.lower(): return 'deletion'
 if 'ins' in token.lower(): return 'insertion'
 m = re.fullmatch(r'([A-Z])(\d+)([A-Z])', token)
 if m: return 'synonymous' if m[1] == m[3] else 'substitution'
 return 'other'

class MutationFeatures(BaseEstimator, TransformerMixin):
 def __init__(self, mode='full', min_df=3):
  self.mode = mode
  self.min_df = min_df

 def dictionaries(self, X):
  genes = list(X.columns)
  for row in X.itertuples(index=False, name=None):
   d = {}; counts = {}; burden = 0; events = 0; positions = []
   changed = set()
   for gene, value in zip(genes, row):
    if value is None or str(value) in ('nan', '', '<NA>'):
     d['missing:' + gene] = 1; continue
    if value == 'WT': continue
    changed.add(gene); burden += 1; d['gene:' + gene] = 1
    if self.mode == 'binary': continue
    tokens = str(value).split()
    for raw_token in tokens:
     token = re.sub(r'^([A-Z]\d+)X$', r'\1*', raw_token)
     events += 1; typ = kind(token); counts[typ] = counts.get(typ, 0) + 1
     d['variant:' + gene + ':' + token] = 1
     d['type:' + gene + ':' + typ] = 1
     m = re.search(r'\d+', token)
     if m:
      position = int(m[0]); positions.append(position)
      d['position:' + gene + ':' + str(position)] = 1
   if self.mode == 'full':
    d['numeric:log_mutated_genes'] = np.log1p(burden)
    d['numeric:log_events'] = np.log1p(events)
    d['numeric:multi_event_excess'] = np.log1p(max(0, events-burden))
    for typ in ['frameshift','stop','deletion','insertion','synonymous','substitution','other']:
     d['numeric:log_' + typ] = np.log1p(counts.get(typ,0))
     d['numeric:fraction_' + typ] = counts.get(typ,0)/max(events,1)
    d['numeric:log_mean_position'] = np.log1p(np.mean(positions)) if positions else 0.
    for name, members in GROUPS.items():
     d['module:' + name] = np.log1p(len(changed.intersection(members)))
   yield d

 def fit(self, X, y=None):
  self.genes_ = list(X.columns)
  self.vectorizer_ = DictVectorizer(dtype=np.float32)
  matrix = self.vectorizer_.fit_transform(self.dictionaries(X))
  names = self.vectorizer_.get_feature_names_out()
  self.keep_ = (np.asarray((matrix != 0).sum(axis=0)).ravel() >= self.min_df) | np.array([s.startswith(('numeric:', 'module:')) for s in names])
  self.feature_names_ = names[self.keep_]
  return self

 def transform(self, X):
  missing = set(self.genes_) - set(X.columns)
  if missing: raise ValueError('Missing gene columns: ' + str(sorted(missing)[:10]))
  return sparse.csr_matrix(self.vectorizer_.transform(self.dictionaries(X[self.genes_]))[:, self.keep_])

 def get_feature_names_out(self, input_features=None): return self.feature_names_
parser=MutationFeatures(mode='full')
# Pure per-row parsing before splits is deterministic; vocabularies are fitted below inside training folds.
records=list(parser.dictionaries(X))
burdens=np.array([sum(k.startswith('gene:') for k in d) for d in records],dtype=np.float32)
class_counts=pd.Series(y).value_counts().rename_axis('SUBCLASS').reset_index(name='count')
class_counts.to_csv(OUT/'class_counts.csv',index=False)
eda=[]
for name,frame in [('train',X),('test',test[genes])]:
 arr=frame.to_numpy();mask=pd.notna(arr)&(arr!='WT');values=arr[mask]
 eda.append({'dataset':name,'rows':len(frame),'median_mutated_genes':float(np.median(mask.sum(axis=1))),'mean_mutated_genes':float(mask.sum(axis=1).mean()),'missing':int(pd.isna(arr).sum()),'multi_token_cells':sum(' ' in str(v) for v in values),'X_stop_tokens':sum(bool(re.fullmatch(r'[A-Z]\d+X',t)) for v in values for t in str(v).split())})
eda=pd.DataFrame(eda);eda.to_csv(OUT/'eda.csv',index=False);display(eda,class_counts)
# Train/test annotations are described only; no test frequency feeds selectors or prototypes.


,dataset,rows,median_mutated_genes,mean_mutated_genes,missing,multi_token_cells,X_stop_tokens
0,train,6201,14.0,35.299629,0,22026,0
1,test,2546,28.0,78.134328,237,63178,14477


,SUBCLASS,count
0,BRCA,786
1,KIPAN,515
2,GBMLGG,461
3,STES,379
4,KIRC,334
5,THCA,324
6,SKCM,276
7,PRAD,266
8,OV,253
9,LGG,229


## 2. 폴드 내 변수 선별 및 상관 제거
변이 후보 빈도(min_df=3)·타깃 연관성(chi2 또는 ANOVA F)·상관계수는 모두 해당 학습 폴드에서만 계산합니다. 타깃 연관성 순위 상위 최대 4,096개 풀에서 절대 Pearson 상관 ≥ 0.95/0.99인 중복 변수를 순위가 높은 하나로 줄입니다. 전체 2만 개의 상관 행렬을 만들지 않습니다. 드문 3명짜리 변수의 큰 연관성은 과적합일 수 있어 규제와 검증으로 비교합니다. 상관 임계값은 생물학적 독립성을 뜻하지 않습니다.

`*_full`은 환자 numeric 요약을 포함하는 대조군입니다. embedded_full은 MaxAbs 스케일링 후 one-vs-rest L1 로지스틱(C=0.2)의 클래스별 최대 절댓값 계수로 공동 연관성을 선별합니다. 최종 분류기 C와 L1 선별기 C는 다른 파라미터입니다. 512개 이상 손실을 확인한 뒤 최대 선택 개수를 2,048개까지 확장했습니다.

In [3]:
def relevance_scores(a,target,method):
 base=method.removesuffix('_full')
 if base=='embedded':
  scaled=csr32(MaxAbsScaler().fit_transform(a))
  fitted=OneVsRestClassifier(LogisticRegression(solver='liblinear',l1_ratio=1,C=.2,class_weight='balanced',max_iter=2000),n_jobs=1).fit(scaled,target)
  return np.max(np.abs(np.vstack([e.coef_[0] for e in fitted.estimators_])),axis=0)
 return (chi2(a,target) if base=='chi2' else f_classif(a,target))[0]

def csr32(a):
 b=sparse.csr_matrix(a,dtype=np.float32)
 if b.nnz>=np.iinfo(np.int32).max:raise ValueError('Sparse matrix too large for tree index type')
 b.indices=b.indices.astype(np.int32,copy=False);b.indptr=b.indptr.astype(np.int32,copy=False)
 return b

class Space:
 def fit(self,rs):
  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(rs)
  names=self.vec.get_feature_names_out();df=np.asarray((a!=0).sum(axis=0)).ravel()
  keep=(df>=3)|np.array([n.startswith(('numeric:','module:')) for n in names])
  self.raw=np.flatnonzero(keep);self.names=names[keep]
  return csr32(a[:,self.raw])
 def transform(self,rs):return csr32(self.vec.transform(rs)[:,self.raw])
 def compact_vec(self,indices):
  v=copy.deepcopy(self.vec);mask=np.zeros(len(v.feature_names_),bool);mask[self.raw[np.sort(indices)]]=True
  v.restrict(mask);return v

class Selector:
 def __init__(self,a,target,names):
  self.a=a;self.y=target;self.names=names;self.cache={};self.corr={};self.scores={};self.audit=[];self.examples=[]
 def order(self,method,threshold):
  key=(method,threshold)
  if key in self.cache:return self.cache[key]
  if method not in self.scores:
   with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning);score=relevance_scores(self.a,self.y,method)
   score=np.nan_to_num(score,nan=0,posinf=np.finfo(np.float32).max,neginf=0)
   if not method.endswith('_full'):score[np.array([n.startswith('numeric:') for n in self.names])]=0
   self.scores[method]=score
  score=self.scores[method]
  if method not in self.corr:
   pool=np.argsort(-score,kind='stable')[:min(4096,int((score>0).sum()))]
   z=self.a[:,pool];mean=np.asarray(z.mean(axis=0)).ravel();var=np.asarray(z.multiply(z).mean(axis=0)).ravel()-mean**2
   sd=np.sqrt(np.maximum(var,1e-12));corr=(z.T@z).toarray()/z.shape[0]-np.outer(mean,mean)
   corr/=np.outer(sd,sd);np.clip(corr,-1,1,out=corr)
   self.corr[method]=(pool,corr)
  pool,corr=self.corr[method];chosen=[]
  for i in range(len(pool)):
   if chosen:
    correlations=np.abs(corr[i,chosen]);j=int(np.argmax(correlations))
    if correlations[j]>=threshold:
     if len(self.examples)<100:self.examples.append({'method':method,'threshold':threshold,'kept':self.names[pool[chosen[j]]],'discarded':self.names[pool[i]],'correlation':float(corr[i,chosen[j]])})
     continue
   chosen.append(i)
   if len(chosen)>=2048:break
  order=pool[chosen];self.cache[key]=order
  self.audit.append({'method':method,'threshold':threshold,'association_pool':len(pool),'remaining_up_to_2048':len(order)})
  return order
 def select(self,method,threshold,k):return np.sort(self.order(method,threshold)[:k])

class CompactPredictor:
 def __init__(self,space,cols,est):
  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out();self.est=est
  self.direct_inputs=len(self.names);self.raw_inputs=len(self.names);self.kind='compact'
 def predict(self,rs):return self.est.predict_proba(csr32(self.vec.transform(rs)))

# Synthetic background: training-only gene frequencies, fixed noise levels 10 and 40.
# Same raw perturbations for every model; raw dicts are re-parsed so global summaries stay consistent.
def background_frame(frame,training_frame,seed,count):
 rng=np.random.default_rng(seed);out=frame.copy();arr=out.to_numpy(copy=True)
 rate=(training_frame.notna()&training_frame.ne('WT')).mean().to_numpy();prob=rate/rate.sum();idx=np.flatnonzero(prob>0)
 for r in range(len(arr)):
  n=min(len(idx),int(rng.poisson(count)))
  for c in rng.choice(len(prob),size=n,replace=False,p=prob):
   token='A1A' if rng.random()<.5 else 'A1V'
   arr[r,c]=token if not isinstance(arr[r,c],str) or arr[r,c]=='WT' else arr[r,c]+' '+token
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)


## 3. 암종별 서명 점수
각 암종의 유전자 변이율을 전체 배경 변이율로 수축 추정(alpha=20)하고, 양의 log 비율과 관측 지지도로 상위 10/25/50개 유전자를 고릅니다. 단순 변이량 증가가 점수를 키우지 않도록 학습 배경에서 기대되는 점수와 분산으로 보정해 26개 점수를 만듭니다.

서명 생성 후 그 점수를 그대로 같은 행의 정답으로 학습하면 자기 정답의 영향을 받을 수 있으므로, **학습 안에서 다시 그룹 3-fold를 나눠 서명 점수를 cross-fit**합니다. 최종 외부 검증에는 외부 학습 전체로 만든 서명을 사용합니다. 선택 변수 128개를 보완하는 버전도 비교합니다. 서명은 데이터에서 학습한 통계적 조합이며 검증된 의학적 경로/진단 기준이 아닙니다.

In [4]:
class SignatureBank:
 def __init__(self,n=25,alpha=20):self.n=n;self.alpha=alpha
 def fit(self,g,target):
  n=len(target);counts=np.bincount(target,minlength=NC).astype(float)
  oh=sparse.csr_matrix((np.ones(n),(target,np.arange(n))),shape=(NC,n))
  ct=(oh@g).toarray();total=np.asarray(g.sum(axis=0)).ravel();bg=np.clip((total+.5)/(n+1),1e-6,1-1e-6)
  pc=(ct+self.alpha*bg)/(counts[:,None]+self.alpha)
  w=np.clip(np.log(np.maximum(pc,1e-9)/bg),0,4)
  support=w*np.log1p(ct);pruned=np.zeros_like(w)
  for c in range(NC):
   order=np.argsort(-support[c],kind='stable')[:self.n]
   order=order[(ct[c,order]>=2)&(support[c,order]>0)]
   pruned[c,order]=w[c,order]
  # A background draw samples genes according to their training mutation frequencies.
  q=bg/bg.sum();mu=pruned@q;var=(pruned**2)@q-mu**2
  self.active=np.flatnonzero(np.any(pruned!=0,axis=0));self.weights=pruned[:,self.active].T.astype(np.float32)
  self.mu=mu.astype(np.float32);self.var=np.maximum(var,1e-6).astype(np.float32)
  return self
 def active_scores(self,gactive,m):
  raw=np.asarray(gactive@self.weights);z=(raw-m[:,None]*self.mu)/np.sqrt(1+m[:,None]*self.var)
  return np.clip(z,-20,20).astype(np.float32)
 def scores(self,g,m):return self.active_scores(g[:,self.active],m)

def crossfit_bank(g,target,grp,m,n):
 z=np.zeros((len(target),NC),np.float32)
 inner=StratifiedGroupKFold(3,shuffle=True,random_state=SEED+71)
 for a,b in inner.split(g,target,grp):
  assert not set(grp[a]).intersection(grp[b])
  bank=SignatureBank(n).fit(g[a],target[a]);z[b]=bank.scores(g[b],m[b])
 return z,SignatureBank(n).fit(g,target)

class SignaturePredictor:
 def __init__(self,space,gene_cols,bank,addon,est,scaler):
  active=gene_cols[bank.active];cols=np.sort(np.unique(np.concatenate([active,addon])))
  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out()
  self.active_pos=np.searchsorted(cols,active);self.addon_pos=np.searchsorted(cols,addon)
  self.bank=bank;self.est=est;self.scaler=scaler
  self.direct_inputs=NC+len(addon);self.raw_inputs=len(cols);self.kind='signature'
  self.signature_names=space.names[active]
 def predict(self,rs):
  a=csr32(self.vec.transform(rs));m=np.array([sum(k.startswith('gene:') for k in d) for d in rs],np.float32)
  z=self.bank.active_scores(a[:,self.active_pos],m)
  if len(self.addon_pos):z=np.hstack([z,a[:,self.addon_pos].toarray()])
  if self.scaler is not None:z=self.scaler.transform(z)
  return self.est.predict_proba(z)

CONFIGS={}
for method in ['chi2','f_classif']:
 for threshold in [.95,.99]:
  for k in [64,128,256,512]:
   for c in [.1,1,10]:
    name=f'lr_{method}_r{threshold}_k{k}_C{c}'
    CONFIGS[name]=dict(family='compact_lr',method=method,r=threshold,k=k,C=c)
for k in [128,512]:
 for leaves in [7,15]:CONFIGS[f'lgb_k{k}_l{leaves}']=dict(family='compact_lgb',method='chi2',r=.95,k=k,leaves=leaves)
 for depth in [12,20]:CONFIGS[f'extra_k{k}_d{depth}']=dict(family='compact_extra',method='chi2',r=.95,k=k,depth=depth)
for n in [10,25,50]:
 for addon in [0,128]:
  for c in [.1,1,10]:CONFIGS[f'sig_n{n}_a{addon}_C{c}']=dict(family='signature_lr',n=n,addon=addon,C=c)
for addon in [0,128]:
 for leaves in [7,15]:CONFIGS[f'sig_lgb_n25_a{addon}_l{leaves}']=dict(family='signature_lgb',n=25,addon=addon,leaves=leaves)
CONFIGS['baseline_full_lr']=dict(family='baseline_lr')
CONFIGS['baseline_full_extra']=dict(family='baseline_extra')
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))
print('Single configurations:',len(CONFIGS),'Development folds:',len(folds))

def make_estimator(cfg):
 f=cfg['family']
 if f.endswith('lr'):
  return LogisticRegression(C=cfg.get('C',1),class_weight='balanced',max_iter=2000)
 if 'lgb' in f:
  return LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=cfg['leaves'],min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
 if f=='baseline_extra':return ExtraTreesClassifier(n_estimators=300,min_samples_leaf=2,max_features=.15,class_weight='balanced',n_jobs=2,random_state=SEED)
 return ExtraTreesClassifier(n_estimators=250,min_samples_leaf=3,max_depth=cfg['depth'],max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)

def train_candidate(name,space,a,target,idx,selector,bank_cache):
 cfg=CONFIGS[name];family=cfg['family']
 if family.startswith('signature'):
  gene_cols=np.flatnonzero(np.array([s.startswith('gene:') for s in space.names]));g=a[:,gene_cols]
  n=cfg['n']
  if n not in bank_cache:bank_cache[n]=crossfit_bank(g,target,groups[idx],burdens[idx],n)
  z,bank=bank_cache[n];addon=selector.select('chi2',.95,cfg['addon']) if cfg['addon'] else np.array([],dtype=int)
  meta=np.hstack([z,a[:,addon].toarray()]) if len(addon) else z
  scaler=StandardScaler().fit(meta) if family=='signature_lr' else None
  fit_meta=scaler.transform(meta) if scaler is not None else meta
  est=make_estimator(cfg).fit(fit_meta,target)
  return SignaturePredictor(space,gene_cols,bank,addon,est,scaler)
 cols=np.arange(a.shape[1]) if family.startswith('baseline') else selector.select(cfg['method'],cfg['r'],cfg['k'])
 est=make_estimator(cfg).fit(a[:,cols],target)
 return CompactPredictor(space,cols,est)

def fit_indices(name,idx):
 rs=[records[i] for i in idx];space=Space();a=space.fit(rs);selector=Selector(a,yi[idx],space.names)
 model=train_candidate(name,space,a,yi[idx],idx,selector,{})
 return model,space,selector

# Refinement after coarse CV: fewer than 512 predictors lost substantial information.
# Include larger still-compact k and global-summary-preserving control arms.
REFINED=[]
for method in ['chi2_full','f_classif_full']:
 for k in [512,1024,2048]:
  for c in [.03,.1,.3,1]:
   name=f'lr_{method}_r0.95_k{k}_C{c}';CONFIGS[name]=dict(family='compact_lr',method=method,r=.95,k=k,C=c);REFINED.append(name)
for method in ['chi2','f_classif']:
 for k in [1024,2048]:
  for c in [.03,.1,.3]:
   name=f'lr_{method}_r0.95_k{k}_C{c}';CONFIGS[name]=dict(family='compact_lr',method=method,r=.95,k=k,C=c);REFINED.append(name)
for k in [512,1024,2048]:
 for leaves in [7,15]:
  name=f'lgb_full_k{k}_l{leaves}';CONFIGS[name]=dict(family='compact_lgb',method='f_classif_full',r=.95,k=k,leaves=leaves);REFINED.append(name)
 for depth in [12,20]:
  name=f'extra_full_k{k}_d{depth}';CONFIGS[name]=dict(family='compact_extra',method='f_classif_full',r=.95,k=k,depth=depth);REFINED.append(name)
for k in [256,512,1024]:
 for c in [.1,.3,1]:
  name=f'lr_embedded_k{k}_C{c}';CONFIGS[name]=dict(family='compact_lr',method='embedded_full',r=.99,k=k,C=c);REFINED.append(name)
for k in [512,1024]:
 for leaves in [7,15]:
  name=f'lgb_embedded_k{k}_l{leaves}';CONFIGS[name]=dict(family='compact_lgb',method='embedded_full',r=.99,k=k,leaves=leaves);REFINED.append(name)
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))
print('Refined total configurations:',len(CONFIGS),'Added:',len(REFINED))


Single configurations: 80 Development folds: 3


## 4. 동일 그룹 3-fold 하이퍼파라미터 비교
각 후보에 동일한 clean·배경 변이 10개·40개 검증 행을 사용합니다. 스트레스는 해당 학습 폴드의 유전자 변이율로만 생성한 합성 민감도 검사이며 실제 테스트 성능의 추정치가 아닙니다.

선택 기준을 고정합니다: `robust_score = 0.5×clean + 0.25×stress10 + 0.25×stress40`의 fold 평균. 입력 축소 목적에 맞춰 compact/signature 후보 안에서 선택합니다. 원래 전체 변수 LR·ExtraTrees 및 두 모델 평균은 별도 기준으로 비교합니다. 축소 계열별 상위 모델 최대 3개의 동일 확률 평균을 1개 추가 비교합니다. 많은 후보 중 최댓값을 고르므로 CV 최댓값에는 선택 편향이 있습니다. 기존 홀드아웃을 하이퍼파라미터 선택에 사용하지 않습니다.

In [5]:
cv_rows=[];OOF={name:np.zeros((3,len(dev),NC),np.float32) for name in CONFIGS}
for fold,(tr,va) in enumerate(folds):
 tr_idx=dev[tr];va_idx=dev[va];training=[records[i] for i in tr_idx]
 views=[[records[i] for i in va_idx]]
 for count in [10,40]:
  noisy=background_frame(X.iloc[va_idx],X.iloc[tr_idx],SEED+count+fold,count)
  views.append(list(parser.dictionaries(noisy)))
 space=Space();a=space.fit(training);selector=Selector(a,yi[tr_idx],space.names);banks={}
 for name,cfg in CONFIGS.items():
  start=time.time();model=train_candidate(name,space,a,yi[tr_idx],tr_idx,selector,banks)
  fs=[]
  for j,view in enumerate(views):
   p=model.predict(view)
   assert p.shape==(len(va_idx),NC) and np.isfinite(p).all()
   OOF[name][j,va]=p;fs.append(f1_score(yi[va_idx],p.argmax(axis=1),labels=np.arange(NC),average='macro',zero_division=0))
  row=dict(model=name,fold=fold,family=cfg['family'],clean=fs[0],stress10=fs[1],stress40=fs[2],robust_score=.5*fs[0]+.25*fs[1]+.25*fs[2],direct_inputs=model.direct_inputs,raw_selected=model.raw_inputs,seconds=time.time()-start)
  cv_rows.append(row);pd.DataFrame(cv_rows).to_csv(OUT/'cv_progress.csv',index=False)
  print(name,'fold',fold,'F1',*[round(v,4) for v in fs],'inputs',model.direct_inputs,'seconds',round(row['seconds'],1),flush=True)
  del model
 pd.DataFrame(selector.audit).to_csv(OUT/f'selector_audit_fold{fold}.csv',index=False)
 pd.DataFrame(selector.examples).to_csv(OUT/f'correlation_examples_fold{fold}.csv',index=False)
cv=pd.DataFrame(cv_rows);cv.to_csv(OUT/'cv_folds.csv',index=False)
ranking=cv.groupby('model').agg(clean=('clean','mean'),stress10=('stress10','mean'),stress40=('stress40','mean'),robust_score=('robust_score','mean'),clean_std=('clean','std'),direct_inputs=('direct_inputs','mean'),raw_selected=('raw_selected','mean')).sort_values('robust_score',ascending=False)

def evaluate_ensemble(names,title):
 p=sum(OOF[n] for n in names)/len(names);scores=[]
 for fold,(_,va) in enumerate(folds):
  f=[f1_score(yi[dev[va]],p[j,va].argmax(axis=1),labels=np.arange(NC),average='macro',zero_division=0) for j in range(3)]
  scores.append(f)
 ss=np.array(scores)
 ranking.loc[title]=[ss[:,0].mean(),ss[:,1].mean(),ss[:,2].mean(),(.5*ss[:,0]+.25*ss[:,1]+.25*ss[:,2]).mean(),ss[:,0].std(ddof=1),sum(ranking.loc[n,'direct_inputs'] for n in names),sum(ranking.loc[n,'raw_selected'] for n in names)]
 return p
baseline_names=['baseline_full_lr','baseline_full_extra'];evaluate_ensemble(baseline_names,'baseline_v3_blend')
compact=list(ranking.index[~ranking.index.str.startswith('baseline')]);family_best=[];seen=set()
for name in compact:
 family=CONFIGS[name]['family']
 if family not in seen:family_best.append(name);seen.add(family)
 if len(family_best)==3:break
compact_blend=evaluate_ensemble(family_best,'compact_family_blend')
ranking=ranking.sort_values('robust_score',ascending=False)
nonbaseline=[n for n in ranking.index if not n.startswith('baseline')]
winner=nonbaseline[0];selected=family_best if winner=='compact_family_blend' else [winner]
ranking.to_csv(OUT/'model_comparison.csv');(OUT/'selection.json').write_text(json.dumps({'winner':winner,'selected':selected,'baseline':baseline_names,'criterion':'.5 clean + .25 stress10 + .25 stress40'},indent=2))
np.savez_compressed(OUT/'oof_probabilities.npz',dev=dev,labels=labels,**OOF)
print('SELECTED COMPACT CANDIDATE:',selected);display(ranking.head(25))


Reusing completed initial-80 OOF predictions; evaluating 61 refinement configurations on identical folds.
lr_chi2_full_r0.95_k512_C0.03 fold 0 F1 0.3702 0.2983 0.156 inputs 512 seconds 1.2
lr_chi2_full_r0.95_k512_C0.1 fold 0 F1 0.3821 0.3018 0.1758 inputs 512 seconds 0.7
lr_chi2_full_r0.95_k512_C0.3 fold 0 F1 0.3767 0.3054 0.1731 inputs 512 seconds 0.7
lr_chi2_full_r0.95_k512_C1 fold 0 F1 0.3683 0.2957 0.1634 inputs 512 seconds 0.8
lr_chi2_full_r0.95_k1024_C0.03 fold 0 F1 0.3859 0.3126 0.1686 inputs 1024 seconds 0.7
lr_chi2_full_r0.95_k1024_C0.1 fold 0 F1 0.3981 0.3159 0.1857 inputs 1024 seconds 0.8
lr_chi2_full_r0.95_k1024_C0.3 fold 0 F1 0.3916 0.32 0.1807 inputs 1024 seconds 0.8
lr_chi2_full_r0.95_k1024_C1 fold 0 F1 0.3789 0.3058 0.177 inputs 1024 seconds 1.0
lr_chi2_full_r0.95_k2048_C0.03 fold 0 F1 0.4038 0.3296 0.1756 inputs 2048 seconds 0.9
lr_chi2_full_r0.95_k2048_C0.1 fold 0 F1 0.411 0.3323 0.195 inputs 2048 seconds 1.0
lr_chi2_full_r0.95_k2048_C0.3 fold 0 F1 0.3982 0.3342 0.196

/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Use

## 5. 기존 홀드아웃 재평가
이 결과를 보고 후보·하이퍼파라미터를 바꾸지 않습니다. 기존부터 여러 번 본 1,241명 평가이므로 독립적인 최종 성능을 주장하지 않습니다. 입력 수를 줄였어도 성능이 반드시 좋아지는 것은 아닙니다.

In [6]:
dev_models=[]
for name in selected:
 m,_,_=fit_indices(name,dev);dev_models.append(m)
hr=[records[i] for i in hold];hp=sum(m.predict(hr) for m in dev_models)/len(dev_models);pred=labels[hp.argmax(axis=1)]
metrics=dict(macro_f1=f1_score(y[hold],pred,average='macro',zero_division=0),accuracy=accuracy_score(y[hold],pred),weighted_f1=f1_score(y[hold],pred,average='weighted',zero_division=0),n=len(hold))
(OUT/'holdout_metrics.json').write_text(json.dumps(metrics,indent=2))
hreport=pd.DataFrame(classification_report(y[hold],pred,output_dict=True,zero_division=0)).T;hreport.to_csv(OUT/'holdout_classification_report.csv')
pd.DataFrame({'ID':train.ID.iloc[hold].values,'true':y[hold],'prediction':pred,'confidence':hp.max(axis=1)}).to_csv(OUT/'holdout_predictions.csv',index=False)
print('Previously observed holdout:',metrics);display(hreport);del dev_models

Previously observed holdout: {'macro_f1': 0.44454416932201135, 'accuracy': 0.4375503626107977, 'weighted_f1': 0.4369990028713108, 'n': 1241}
              precision    recall  f1-score     support
ACC            0.750000  0.857143  0.800000    14.00000
BLCA           0.411765  0.350000  0.378378    20.00000
BRCA           0.728261  0.426752  0.538153   157.00000
CESC           0.315789  0.387097  0.347826    31.00000
COAD           0.730769  0.863636  0.791667    44.00000
DLBC           1.000000  0.714286  0.833333     7.00000
GBMLGG         0.512821  0.217391  0.305344    92.00000
HNSC           0.342105  0.295455  0.317073    44.00000
KIPAN          0.408163  0.194175  0.263158   103.00000
KIRC           0.365591  0.507463  0.425000    67.00000
LAML           0.478261  0.354839  0.407407    31.00000
LGG            0.514706  0.760870  0.614035    46.00000
LIHC           0.480000  0.387097  0.428571    31.00000
LUAD           0.312500  0.277778  0.294118    36.00000
LUSC           0.36

/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)


## 6. 전체 학습, 실제 변수·서명 저장, 피클·제출 생성
선택한 설정만 train 전체로 재학습합니다. 최종 vectorizer는 실제 필요한 변수만 남겨 저장합니다. 서명 모델은 선택한 driver 유전자와 추가 선택 변수만 저장하고, 전체 변이 유전자 수는 배경 보정용 스칼라로 계산합니다. 분류기 직접 입력 수와 서명 생성용 원시 유전자 수를 구분해서 기록합니다. 새 제출 파일은 이전 파일을 덮어쓰지 않습니다.

In [7]:
models=[];feature_rows=[];signature_rows=[];selector_audits=[];model_sizes=[]
for name in selected:
 m,space,selector=fit_indices(name,np.arange(len(train)));models.append(m)
 method=CONFIGS[name].get('method','signature')
 score=selector.scores.get(method)
 for feature_index,f in enumerate(m.names):
  original_index=int(np.searchsorted(space.names,f))
  importance=float(np.abs(m.est.coef_[:,feature_index]).mean()) if m.kind=='compact' and hasattr(m.est,'coef_') else (float(m.est.feature_importances_[feature_index]) if m.kind=='compact' and hasattr(m.est,'feature_importances_') else None)
  feature_rows.append({'model':name,'feature':f,'family':f.split(':')[0],'selection_method':method,'target_relevance':float(score[original_index]) if score is not None else None,'model_importance':importance})
 if selector.examples:pd.DataFrame(selector.examples).to_csv(OUT/(name+'_correlated_examples.csv'),index=False)
 if isinstance(m,SignaturePredictor):
  for j,c in enumerate(labels):
   for f,w in zip(m.signature_names,m.bank.weights[:,j]):
    if w>0:signature_rows.append({'model':name,'SUBCLASS':c,'feature':f,'signature_weight':float(w)})
 param_count=int(m.est.coef_.size+m.est.intercept_.size) if hasattr(m.est,'coef_') else None
 model_sizes.append({'model':name,'direct_classifier_inputs':m.direct_inputs,'raw_selected_variables':m.raw_inputs,'linear_parameter_count':param_count,'config':CONFIGS[name]})
 selector_audits.extend([dict(model=name,**r) for r in selector.audit])
features=pd.DataFrame(feature_rows);features.to_csv(OUT/'final_features.csv',index=False)
pd.DataFrame(signature_rows,columns=['model','SUBCLASS','feature','signature_weight']).to_csv(OUT/'class_signatures.csv',index=False)
pd.DataFrame(selector_audits).to_csv(OUT/'selector_audit_full_train.csv',index=False)
(OUT/'model_sizes.json').write_text(json.dumps(model_sizes,indent=2))

def predict_bundle(bundle,frame):
 parsed=list(MutationFeatures(mode='full').dictionaries(frame[bundle['genes']]))
 return sum(m.predict(parsed) for m in bundle['models'])/len(bundle['models'])
bundle={'version':4,'models':models,'selected':selected,'genes':genes,'classes':labels,'holdout_metrics':metrics,'configs':{n:CONFIGS[n] for n in selected},'model_sizes':model_sizes,'predict':predict_bundle}
model_path=OUT/'cancer_model_v4.pkl'
with model_path.open('wb') as f:cloudpickle.dump(bundle,f,protocol=pickle.HIGHEST_PROTOCOL)
with model_path.open('rb') as f:loaded=pickle.load(f)
tp=loaded['predict'](loaded,test)
assert np.isfinite(tp).all() and np.allclose(tp.sum(axis=1),1)
submission=sample.copy();submission.SUBCLASS=labels[tp.argmax(axis=1)]
assert submission.ID.equals(sample.ID) and submission.SUBCLASS.isin(labels).all()
submission.to_csv(OUT/'submission_v4.csv',index=False);submission.to_csv(SUBMIT/'혜진_submission_v4.csv',index=False)
pd.DataFrame(tp,index=test.ID,columns=labels).to_csv(OUT/'test_probabilities.csv')
print('MODEL',model_path.resolve(),'MiB',round(model_path.stat().st_size/1024**2,2));display(FileLink(str(model_path)))
print('SUBMISSION',(SUBMIT/'혜진_submission_v4.csv').resolve());display(pd.DataFrame(model_sizes).drop(columns='config'))


MODEL /Users/mac/project/cancer_project/artifacts/v4/cancer_model_v4.pkl MiB 40.28
/Users/mac/project/cancer_project/artifacts/v4/cancer_model_v4.pkl
SUBMISSION /Users/mac/project/cancer_project/submission/혜진_submission_v4.csv
                      model  ...  linear_parameter_count
0  lr_chi2_r0.95_k2048_C0.1  ...                 53274.0
1        lgb_full_k2048_l15  ...                     NaN
2      extra_full_k1024_d20  ...                     NaN

[3 rows x 4 columns]


/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/Users/mac/project/cancer_project/.venv/lib/python3.15/site-packages/sklearn/feature_selection/_univariate_selection.py:110: UserWarning: Features [0 0] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)


## 7. 보고서·독립 로딩·결과 검증
새 대회 점수는 제출 후에만 확인합니다. 로컬 점수와 모델 크기 개선을 대회 점수 상승으로 혼동하지 않습니다. 실험 전체 결과, 실제 최종 변수, 암종별 서명과 선택 설정을 저장합니다.

In [8]:
import subprocess,tempfile,importlib.metadata
program="""
import pickle,sys,json,pandas as pd,numpy as np
with open(sys.argv[1],'rb') as f:b=pickle.load(f)
d=pd.read_csv(sys.argv[2],nrows=4);p=b['predict'](b,d)
assert p.shape==(len(d),26) and np.isfinite(p).all() and np.allclose(p.sum(axis=1),1)
assert not {'mutation_model','report_writer'}.intersection(sys.modules)
print(json.dumps({'status':'passed','rows':len(d),'version':b['version']}))
"""
env=os.environ.copy();env.pop('PYTHONPATH',None)
with tempfile.TemporaryDirectory() as td:
 result=subprocess.run([sys.executable,'-c',program,str(model_path.resolve()),str((DATA/'test.csv').resolve())],cwd=td,env=env,text=True,capture_output=True,check=True)
check=json.loads(result.stdout)
assert np.allclose(tp[:4],loaded['predict'](loaded,test.iloc[:4]))
assert not set(groups[dev]).intersection(groups[hold])
versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','scipy','scikit-learn','lightgbm','matplotlib','cloudpickle','threadpoolctl','ipython']}
(WORK/'requirements-v4.txt').write_text('\n'.join(p+'=='+v for p,v in versions.items())+'\n')
metadata={'status':'passed','version':4,'selected':selected,'model_sizes':model_sizes,'holdout_metrics':metrics,'new_public_score':None,'prior_public_scores':{'v1':0.2765425197,'v3':0.2998780222},'independent_loading':check,'model_bytes':model_path.stat().st_size,'submission_rows':len(submission),'packages':versions,'python':sys.version,'sha256':{str(p.name):hashlib.sha256(p.read_bytes()).hexdigest() for p in [DATA/'train.csv',DATA/'test.csv',model_path,SUBMIT/'혜진_submission_v4.csv']}}
(OUT/'verification.json').write_text(json.dumps(metadata,indent=2))

def mdtable(df):
 return '| '+' | '.join(map(str,df.columns))+' |\n| '+' | '.join(['---']*len(df.columns))+' |\n'+'\n'.join('| '+' | '.join(f'{v:.5f}' if isinstance(v,float) else str(v) for v in row)+' |' for row in df.itertuples(index=False,name=None))
nonbase=ranking.loc[[n for n in ranking.index if not n.startswith('baseline')]]
summary=pd.DataFrame(model_sizes).drop(columns='config')
text=f"""# 혜진 v4 변수 축소·암종 서명 실험 보고서

## 결론과 실제 성능 구분
기존 v1 Public 0.2765425197, v3 Public **0.2998780222**. v4 Public은 **아직 미측정**이다. 새 대회 성능 개선을 주장하지 않는다.
선택 후보: {', '.join(selected)}. 선택은 개발 그룹 CV의 clean/stress10/stress40에 0.5/0.25/0.25를 준 점수로만 했다. 입력 축소라는 실험 목적에 맞춰 baseline 전체 변수 모델은 비교 기준으로만 사용하고 compact/signature 후보에서 선정했다. 축소 모델의 로컬/스트레스 성능이 기준보다 나쁜 경우도 표에 그대로 남긴다.

기존부터 관측된 홀드아웃: {json.dumps(metrics,ensure_ascii=False)}. 여러 실험에서 이미 관측된 데이터이므로 독립적인 최종 검증은 아니다. 이 점수를 보고 선택 설정을 바꾸지 않았다. 이전 v3의 같은 홀드아웃 macro F1은 0.4237006861이었다.

## 입력 변수와 하이퍼파라미터의 차이
21,448개는 v3의 입력 변수 수였다. k, 연관성 점수 방식, Pearson 중복 임계값 r, 규제 강도 C, leaf/depth는 하이퍼파라미터다. v4는 학습 부분에서만 고른 64/128/256/512/1024/2048개 또는 cross-fit 암종 점수 26개(+변수 128개)를 분류기에 넣는다. 모델의 실제 입력 수와 원시 서명 변수 수를 아래에 구분한다.

{mdtable(summary)}

서명 분류기의 입력이 26개라도 그 점수를 만드는 원시 driver 유전자는 더 많다. linear_parameter_count는 선형 분류기의 계수와 절편 개수만 센 것이며, 전처리 추정량이나 트리 파라미터와 같은 기준이 아니다. 모델 피클 크기는 {model_path.stat().st_size/1024**2:.2f} MiB다.

## 데이터 및 EDA
train 6,201명, test 2,546명, 유전자 4,384개, 클래스 26개. ID는 정렬/식별에만 사용하고 SUBCLASS는 입력 제외한다. WT는 변이 없음 표기, 결측은 변이로 세지 않는다. 변이량은 측정 범위의 변이 유전자 수이며 임상 TMB가 아니다.

{mdtable(eda)}

## 후보 파생변수 사전
| 변수 | 정의 | 근거·한계 |
| --- | --- | --- |
| gene:G | G의 비-WT 여부 | 유전자 수준 신호, 변이 종류 소실 |
| type:G:K | 치환/동의적/종결/frameshift/삽입/결실/기타 존재 | 기능 손상 확정이 아닌 표기 분류 |
| variant:G:T | X→*로 통일한 정확한 토큰 | hotspot 포착, 전사체·희귀 변이에 취약 |
| position:G:N | 같은 유전자의 숫자 위치 | 같은 위치 치환 공유, 유전체 좌표 아님 |
| numeric:* | 환자 변이 수/토큰 수/유형별 개수·비율/위치 요약 | v3 비교 기준에만 포함, stable 선택 풀에서는 제외, *_full 대조군에는 포함 |
| module:* | 7개 수작업 유전자 그룹의 log 변이 수 | 경로 활성도 아님, 학습 내 연관성 검증 후 선택될 수 있음 |
| signature:CLASS | 배경 보정 암종별 양의 log 변이율 점수 | 클래스 서명, 인과성/임상 진단 점수 아님 |

## 변수 선택·중복 제거 방법
1. 폴드 학습 행으로 DictVectorizer 및 출현 환자 3명 기준을 적합한다. 테스트 단어 사전/빈도는 사용하지 않는다.
2. 같은 학습 행의 정답에 대한 chi2 또는 ANOVA F 순위를 계산한다. stable 선택에서는 numeric 전역 요약을 제외하고 *_full에서는 유지한다. embedded_full은 MaxAbs 스케일링 후 OVR L1 로지스틱(C=0.2)의 최대 절댓값 계수로 공동 연관성을 선별한다. L1 선별기와 최종 분류기의 규제 강도는 별개다.
3. 상위 최대 4,096개 풀에서 절대 Pearson 상관이 0.95/0.99 이상인 변수를 중복으로 보고 높은 연관성 순위 하나를 남긴다. 파라미터 r도 개발 CV로 비교한다. 동일 유전자라도 상관이 낮은 서로 다른 변이는 남을 수 있다.
4. 상관 제거 순위의 앞 k개(64/128/256/512/1024/2048)를 선택하고 규제 C=0.1/1/10, 트리 잎/깊이를 비교한다. 실제 생존 변수가 부족하면 k보다 적을 수 있다.
5. 타깃과의 연관성이 높은 순위는 예측 유용성의 가설이며 인과성이나 모든 분포에서의 안정성을 보장하지 않는다. 드문 특성의 우연한 연관성, 다중 비교, 클래스 불균형 영향을 받는다.

선택/상관 계산에 검증·홀드아웃·테스트 행을 적합하지 않았다. 내부 서명 cross-fit에서는 외부 학습으로 정의한 유전자 사전을 공유하지만, 각 내부 검증 행의 정답을 서명 추정에 사용하지 않는다. 외부 검증은 전처리·서명·모델 어느 적합에도 참여하지 않는다.

## 암종 서명 방식
배경 변이율 bg, 클래스별 관측 수 ct와 샘플 수 nc에 alpha=20 수축을 적용해 pc=(ct+alpha*bg)/(nc+alpha)로 추정한다. 양의 log(pc/bg)를 0~4로 제한하고 관측 지지도로 상위 10/25/50 유전자를 선택한다. 배경 기대치와 분산을 빼고 나눠 변이량에 따른 기본 점수 증가를 완화한다. 이는 독립 배경 가정의 근사이며 실제 유전자 공변이와 기술적 노이즈를 완전히 제거하지 못한다.
학습 행의 자기 정답 누수를 줄이기 위해 그룹 내부 3-fold로 서명 점수를 cross-fit하고, 그 점수에 작은 LR/LightGBM을 학습한다. 외부 검증과 최종 추론에서는 학습 전체의 서명을 사용한다. 실제 서명 유전자/가중치는 class_signatures.csv에 기록한다.

## 비교 범위와 결과
초기 80개 설정을 평가한 뒤 개발 CV에서 과도한 축소의 손실을 확인해 61개를 추가했다. 최종 단일 설정 {len(CONFIGS)}개를 동일 외부 그룹 3-fold에서 평가했다. 순차 탐색도 같은 개발 데이터에 적응하므로 선택 편향을 키울 수 있다. 전체 변수 v3 LR/ExtraTrees 및 평균도 같은 분할과 노이즈에서 비교했다. 축소 계열별 상위 최대 3개 평균을 1개 추가 비교했다. 같은 CV에서 많은 후보와 앙상블을 고르므로 최댓값은 선택 편향이 있다. 새로운 Public은 반드시 별도로 측정해야 한다.

{mdtable(ranking.reset_index())}

## 스트레스 검증
검증 샘플에 배경 변이 10/40개를 Poisson으로 추가한다. 유전자 확률은 학습 폴드의 변이 빈도만 사용하고, 동일한 raw 노이즈를 모든 모델에 넣고 요약 변수까지 다시 계산한다. 노이즈는 A1A/A1V placeholder이며 실제 테스트 주석 방식이나 생물학적 변이 분포를 재현하지 못한다. 실제 테스트 정답/분포를 이용한 적합은 없고, 스트레스 점수는 대회 점수의 대체값이 아니다. 증강 학습은 이번 비교에 사용하지 않았다.

## 기존 홀드아웃 클래스별 결과
{mdtable(hreport.reset_index())}

## 저장물·재현
노트북 Run All로 전 과정을 실행한다. artifacts/v4/cv_folds.csv 및 model_comparison.csv는 전체 비교, configurations.json은 설정, final_features.csv는 실제 저장 변수, class_signatures.csv는 클래스별 driver 가중치, model_sizes.json은 입력/계수 수, verification.json은 환경·해시·로딩 검증이다. 피클은 전처리 코드까지 포함하고 bundle['predict'](bundle, dataframe)으로 추론한다. 원본 제출과 v3는 보존하고 submission/혜진_submission_v4.csv로 저장한다. 같은 라이브러리 환경은 requirements-v4.txt에 기록한다.

## 한계
기존 홀드아웃 재사용, 개발 CV에서 많은 후보 선택, 드문 클래스 표본 부족, 원본 프로파일 중복/상충 라벨, 학습·테스트 주석 및 변이량 차이, 합성 노이즈의 현실성 한계가 있다. v3보다 입력이 작아졌다는 사실만으로 새 대회 점수가 높아진 것은 아니다. 외부 데이터·테스트 적합·의사 라벨은 사용하지 않았다.
"""
(WORK/'analysis_report_v4.md').write_text(text)
# Full final-variable dictionary, without invented gene functional annotations.
fd=features.copy();fd['meaning']=fd['family'].map({'gene':'유전자 변이 존재','variant':'정규화된 정확한 변이','position':'유전자 내 숫자 위치','type':'유전자별 변이 유형','module':'수작업 유전자 그룹 집계','numeric':'환자 단위 요약'}).fillna('서명 생성/추가 입력')
(WORK/'selected_variables_v4.md').write_text('# v4 실제 저장 변수 사전\n\n서명 모델의 names는 분류기 입력 26개와 다르며, 서명 생성에 필요한 원시 유전자 및 추가 변수의 합집합입니다. class_signatures.csv에서 클래스별 용도를 확인하세요. 유전자 기능 주석은 원본에 없어 추정하지 않습니다.\n\n'+mdtable(fd))
print('Report',WORK/'analysis_report_v4.md','Standalone check',check,'holdout',metrics)


Report /Users/mac/project/cancer_project/analysis_report_v4.md Standalone check {'status': 'passed', 'rows': 4, 'version': 4} holdout {'macro_f1': 0.44454416932201135, 'accuracy': 0.4375503626107977, 'weighted_f1': 0.4369990028713108, 'n': 1241}


## 8. 최종 파일·변수 합집합·전체 테스트 재로딩 검증
실제 저장된 변수 사전과 전체 테스트 예측을 검증하고, 이전 기준과 달라진 성능을 유리한 결과·불리한 결과 모두 보고서에 기록합니다.

In [9]:
import json,pickle,hashlib
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display,FileLink
OUT=WORK/'artifacts'/'v4'
with (OUT/'cancer_model_v4.pkl').open('rb') as f:
    independent_bundle=pickle.load(f)
independent_p=independent_bundle['predict'](independent_bundle,test)
submitted=pd.read_csv(SUBMIT/'혜진_submission_v4.csv')
stored_p=pd.read_csv(OUT/'test_probabilities.csv',index_col=0)
assert submitted.ID.equals(sample.ID) and submitted.ID.equals(test.ID)
assert np.array_equal(submitted.SUBCLASS.to_numpy(),independent_bundle['classes'][independent_p.argmax(axis=1)])
assert list(stored_p.columns)==list(independent_bundle['classes']) and list(stored_p.index)==list(test.ID)
assert np.allclose(independent_p,stored_p.to_numpy()) and np.allclose(independent_p.sum(axis=1),1)
assert np.isfinite(independent_p).all() and not submitted.isna().any().any()
manifest=pd.read_csv(OUT/'split_manifest.csv',dtype={'group':str})
assert manifest.groupby('group')['split'].nunique().max()==1
final_features=pd.read_csv(OUT/'final_features.csv')
unique_variables=int(final_features.feature.nunique())
sizes=json.loads((OUT/'model_sizes.json').read_text())
checks=json.loads((OUT/'verification.json').read_text())
checks.update(full_independent_test_prediction='passed',train_profile_split_overlap=0,unique_raw_selected_variables=unique_variables,source_notebook='cencer_v4.ipynb')
(OUT/'verification.json').write_text(json.dumps(checks,indent=2))
# Model input count is not a hyperparameter count; keep raw and derived widths distinct.
summary=pd.DataFrame(sizes).drop(columns='config')
print('Verification passed. Unique raw selected variables:',unique_variables)
display(summary)
print('New submission:',(SUBMIT/'혜진_submission_v4.csv').resolve())
display(FileLink(str(SUBMIT/'혜진_submission_v4.csv')))
# All-gene dictionary: retain raw column statistics and update actual selected usage.
train_mask=train[genes].notna()&train[genes].ne('WT')
test_mask=test[genes].notna()&test[genes].ne('WT')
counts=Counter(f.split(':')[1] for f in final_features.feature if f.split(':')[0] in {'gene','variant','position','type','missing'})
selected_modules={f.split(':',1)[1] for f in final_features.feature if f.startswith('module:')}
num_selected=final_features[final_features.family=='numeric'][['model','feature']]
dictionary=[]
for g in genes:
    dictionary.append({'gene':g,'train_mutated':int(train_mask[g].sum()),'train_rate':float(train_mask[g].mean()),'test_rate':float(test_mask[g].mean()),'selected_raw_feature_uses':counts[g],'selected_modules':', '.join(k for k,v in GROUPS.items() if k in selected_modules and g in v) or '-'})
gene_dict=pd.DataFrame(dictionary);gene_dict.to_csv(OUT/'gene_dictionary_v4.csv',index=False)
def _table4(df):
    return '| '+' | '.join(map(str,df.columns))+' |\n| '+' | '.join(['---']*len(df.columns))+' |\n'+'\n'.join('| '+' | '.join(f'{v:.5f}' if isinstance(v,float) else str(v) for v in row)+' |' for row in df.itertuples(index=False,name=None))
(WORK/'gene_dictionary_v4.md').write_text('# v4 전체 유전자 컬럼 사전\n\n각 행은 원본 유전자 컬럼입니다. 값은 WT/변이 표기이며 생물학적 기능 주석은 제공되지 않아 추정하지 않습니다. selected_raw_feature_uses는 최종 모델들의 gene/variant/position/type 변수에 연결된 횟수의 합입니다(모델 간 중복 포함). 0이어도 환자 전체 요약값이나 선택된 유전자 그룹 집계에는 기여할 수 있습니다. 원본 컬럼 수와 분류기 직접 입력 수는 다릅니다. test_rate는 기술통계이며 변수 선택·모델 적합에 사용하지 않았습니다. 변수 접두사와 선택 근거는 analysis_report_v4.md 및 selected_variables_v4.md를 참고하세요.\n\n'+_table4(gene_dict))
# Scientific export figure, using only stored development CV measurements.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
ranked=pd.read_csv(OUT/'model_comparison.csv',index_col=0)
fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,metric,title in zip(axes,['clean','robust_score'],['Clean group-CV Macro F1','Mean of clean / synthetic stress']):
    normal=ranked.loc[~ranked.index.str.startswith('baseline')]
    ax.scatter(normal.direct_inputs,normal[metric],s=15,alpha=.4,label='Compact candidates')
    base=ranked.loc['baseline_v3_blend'];ax.scatter(base.direct_inputs,base[metric],s=70,marker='s',label='V3 full-variable ensemble')
    chosen=ranked.loc[json.loads((OUT/'selection.json').read_text())['winner']]
    ax.scatter(chosen.direct_inputs,chosen[metric],s=120,marker='*',label='Selected compact candidate')
    ax.set_xscale('log');ax.set_xlabel('Sum of classifier input widths (CV mean)');ax.set_ylabel('Macro F1');ax.set_title(title);ax.legend(fontsize=8)
fig.tight_layout();fig.savefig(OUT/'selection_performance.png',dpi=160);plt.close(fig)
# Report the actual mixed result, including unfavorable CV comparisons.
chosen_info=json.loads((OUT/'selection.json').read_text());win=ranked.loc[chosen_info['winner']];base=ranked.loc['baseline_v3_blend']
current_metrics=json.loads((OUT/'holdout_metrics.json').read_text())
ex=[]
for name in independent_bundle['selected']:
    file=OUT/(name+'_correlated_examples.csv')
    if file.exists():
        df=pd.read_csv(file).head(5);df.insert(0,'model',name);ex.append(df)
examples=pd.concat(ex,ignore_index=True) if ex else pd.DataFrame(columns=['model','kept','discarded','correlation'])
marker='\n\n## 최종 검증 및 변수 축소 해석\n'
addendum=f"""
최종 원시 변수의 합집합은 **{unique_variables:,}개**입니다. 모델별 입력 슬롯은 {', '.join(str(s['direct_classifier_inputs']) for s in sizes)}개로, v3의 모델당 21,448개와 구분합니다. 합집합은 같은 원시 변수를 여러 모델이 공유하면 한 번만 셉니다. 원시 변수가 선택되지 않아도 환자 전체 집계에 영향을 줄 수 있습니다.

동일 개발 CV의 v3 ensemble clean F1 **{base.clean:.5f}**, v4 **{win.clean:.5f}**, 가중 스트레스 점수는 v3 **{base.robust_score:.5f}**, v4 **{win.robust_score:.5f}**입니다. 따라서 CV 전반의 개선을 주장할 수 없습니다. 반면 이전부터 관측된 홀드아웃 F1은 0.4237006861에서 **{current_metrics['macro_f1']:.5f}**, 정확도는 **{current_metrics['accuracy']:.5f}**였습니다. 홀드아웃은 반복 관측되었고 많은 후보를 탐색했으므로 이 개선 역시 새로운 데이터의 성능 보장이 아닙니다. 새 Public 점수는 미측정입니다.

암종은 범주이므로 숫자 라벨과의 단순 Pearson 상관으로 순위를 매기지 않았습니다. chi2/ANOVA F 또는 학습 내 L1 분류기 계수로 타깃 연관성을 구하고, Pearson은 입력 변수끼리의 중복 제거에만 사용했습니다. below selected numeric variables는 요약 유지 대조군에서 살아남은 변수이며 변이량/주석 차이의 영향을 받을 수 있습니다.

### 최종 설정
{_table4(summary)}

### 선택된 환자 요약 변수
{_table4(num_selected)}

### 높은 입력 상관 때문에 제외된 변수 예시
{_table4(examples)}

절대 상관 임계값을 넘은 변수 중 높은 타깃 연관성 순위를 남긴 예시입니다. 두 변수의 생물학적 동일성을 증명하는 것은 아닙니다. 최종 변수별 타깃 점수·모델 중요도는 artifacts/v4/final_features.csv에, 전체 원본 컬럼은 gene_dictionary_v4.md에 있습니다. 모델 간 계수/트리 중요도를 직접 비교하지 않습니다.

![변수 수와 개발 검증 성능](artifacts/v4/selection_performance.png)

그림의 x축은 CV에서 각 분류기 입력 수를 더한 값이고, 최종 전체 학습 변수의 합집합과는 다릅니다. 검증 데이터 크기가 작으면 빈도 필터 후 전체 변수 수도 최종 전체 학습보다 작습니다. 전체 테스트 독립 재로딩·CSV/확률 일치·ID 정렬·그룹 분리 검증을 통과했습니다. 기존 제출을 덮어쓰지 않고 혜진_submission_v4.csv로 저장했습니다.
"""
report_path=WORK/'analysis_report_v4.md'
report_path.write_text(report_path.read_text().split(marker)[0]+marker+addendum)
(WORK/'analysis_report.md').write_text(report_path.read_text())
(WORK/'gene_dictionary.md').write_text((WORK/'gene_dictionary_v4.md').read_text())
print('Full test reload, model outputs, variable dictionaries, and report verification passed.')


Verification passed. Unique raw selected variables: 2063
New submission: /Users/mac/project/cancer_project/submission/혜진_submission_v4.csv
Full test reload, model outputs, variable dictionaries, and report verification passed.


Matplotlib is building the font cache; this may take a moment.


,model,direct_classifier_inputs,raw_selected_variables,linear_parameter_count
0,lr_chi2_r0.95_k2048_C0.1,2048,2048,53274.0
1,lgb_full_k2048_l15,2048,2048,NaN
2,extra_full_k1024_d20,1024,1024,NaN


/Users/mac/project/cancer_project/submission/혜진_submission_v4.csv